# Transformers — Kiến trúc làm thay đổi cả NLP

## 1. Vì sao Transformer ra đời?

Trước 2017, NLP chủ yếu dùng RNN/LSTM. Hai vấn đề:

1. **Khó song song hoá.** RNN xử lý tuần tự — token thứ $t$ phụ thuộc trạng thái ẩn của token $t-1$, nên không thể tính cùng lúc cho cả câu. GPU rất giỏi tính song song nhưng RNN lại không khai thác được.
2. **Vẫn khó nhớ phụ thuộc xa.** Dù LSTM khá hơn RNN, với câu dài hàng trăm từ vẫn bị hạn chế.

Năm 2017, paper *"Attention Is All You Need"* (Vaswani et al.) đề xuất kiến trúc **Transformer** — bỏ hoàn toàn cấu trúc hồi quy, chỉ dùng cơ chế **self-attention**. Từ đó:
- Train được trên dữ liệu cực lớn nhờ song song hoá tốt.
- BERT (2018), GPT-2 (2019), GPT-3 (2020), GPT-4, Claude... tất cả đều là Transformer.
- Transformer còn lan sang vision (ViT), reinforcement learning (Decision Transformer), v.v.

Bài này không yêu cầu tự viết Transformer hoàn chỉnh từ đầu (việc đó có thể là một học kỳ riêng), mà ta sẽ:
- Hiểu **self-attention** — trái tim của Transformer.
- Code self-attention thủ công bằng PyTorch để "sờ tận tay".
- Dùng `nn.TransformerEncoderLayer` để build một classifier phân loại văn bản đơn giản.
- Bài tập về nhà: dùng pre-trained BERT (HuggingFace) để fine-tune trên một bài thật.

## 2. Self-attention — ý tưởng cốt lõi

Tưởng tượng câu *"Con mèo đang ngủ trên ghế vì nó mệt"*. Khi đọc đến từ "nó", não ta tự động liên kết "nó" với "con mèo" (chủ thể gần nhất hợp ngữ cảnh). Self-attention chính thức hoá quá trình này.

### Ý tưởng
Cho một câu gồm $n$ token, mỗi token được biểu diễn bởi một vector $x_i \in \mathbb{R}^d$. Với mỗi token $i$, ta muốn tính ra một vector mới $z_i$ — là *trung bình có trọng số* của tất cả các token khác trong câu, trọng số càng cao với những token "liên quan" đến $i$.

Để làm được, ta tạo 3 vector cho mỗi token, gọi là **Query**, **Key**, **Value** (viết tắt Q, K, V):
$$Q_i = W_Q x_i, \quad K_i = W_K x_i, \quad V_i = W_V x_i$$

Trong đó $W_Q, W_K, W_V$ là 3 ma trận trọng số được học.

Trực giác:
- **Query**: token $i$ "hỏi" cái gì.
- **Key**: token $j$ "có gì để chào hàng".
- **Value**: nội dung thực sự của token $j$.

### Công thức Attention
$$
\text{Attention}(Q, K, V) = \text{softmax}\!\Big(\frac{Q K^\top}{\sqrt{d_k}}\Big) V
$$

Bóc tách:
1. $Q K^\top$: ma trận tương đồng — phần tử $(i, j)$ là tích vô hướng giữa Query của token $i$ và Key của token $j$. Cao = liên quan, thấp = không liên quan.
2. Chia $\sqrt{d_k}$: scale để gradient ổn định khi $d_k$ lớn (nếu không, dot product có thể quá to → softmax bão hoà).
3. `softmax`: chuẩn hoá thành phân phối xác suất theo từng hàng.
4. Nhân với $V$: tổng có trọng số các Value.

Kết quả $z_i$ là biểu diễn mới của token $i$ — đã tích hợp thông tin từ các token khác theo tỷ lệ "chú ý".

### Vì sao có $\sqrt{d_k}$?
Nếu $Q, K$ là vector ngẫu nhiên có entries i.i.d. mean 0 var 1, thì dot product $Q \cdot K$ có variance $d_k$ — càng to khi $d_k$ lớn. Chia $\sqrt{d_k}$ đưa variance về 1, giữ softmax không quá nhọn.

### Sơ đồ luồng self-attention

Vẽ ngay đây cho dễ hình dung:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch

fig, ax = plt.subplots(figsize=(11, 5))
ax.set_xlim(0, 11); ax.set_ylim(0, 5); ax.axis('off')

def box(x, y, w, h, text, color='#cce5ff'):
    p = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.05',
                       facecolor=color, edgecolor='black', linewidth=1.2)
    ax.add_patch(p)
    ax.text(x + w/2, y + h/2, text, ha='center', va='center', fontsize=11)

def arrow(x1, y1, x2, y2):
    a = FancyArrowPatch((x1, y1), (x2, y2), arrowstyle='->',
                        mutation_scale=15, color='black', linewidth=1.2)
    ax.add_patch(a)

# Input X
box(0.2, 2.0, 1.0, 1.0, 'X\n(token)', '#fff3cd')

# Three projections
box(2.0, 3.5, 1.0, 0.7, '$W_Q$', '#d4edda')
box(2.0, 2.15, 1.0, 0.7, '$W_K$', '#d4edda')
box(2.0, 0.8, 1.0, 0.7, '$W_V$', '#d4edda')
arrow(1.2, 2.7, 2.0, 3.85); arrow(1.2, 2.5, 2.0, 2.5); arrow(1.2, 2.3, 2.0, 1.15)

# Q, K, V
box(3.5, 3.5, 0.7, 0.7, 'Q', '#cce5ff')
box(3.5, 2.15, 0.7, 0.7, 'K', '#cce5ff')
box(3.5, 0.8, 0.7, 0.7, 'V', '#cce5ff')
arrow(3.0, 3.85, 3.5, 3.85); arrow(3.0, 2.5, 3.5, 2.5); arrow(3.0, 1.15, 3.5, 1.15)

# QK^T / sqrt(d)
box(4.7, 2.6, 1.6, 1.0, '$\\frac{QK^T}{\\sqrt{d_k}}$', '#f8d7da')
arrow(4.2, 3.85, 4.7, 3.3); arrow(4.2, 2.5, 4.7, 2.9)

# Softmax
box(6.6, 2.6, 1.3, 1.0, 'softmax', '#f5c6cb')
arrow(6.3, 3.1, 6.6, 3.1)

# Multiply with V
box(8.2, 1.7, 1.5, 1.6, 'attention\n× V', '#d1ecf1')
arrow(7.9, 3.1, 8.2, 2.7); arrow(4.2, 1.15, 8.2, 1.9)

# Output
box(9.9, 2.0, 1.0, 1.0, 'Z\n(output)', '#fff3cd')
arrow(9.7, 2.5, 9.9, 2.5)

ax.set_title('Self-Attention: từ X tạo Q, K, V → tính attention → tổng có trọng số V', fontsize=12)
plt.tight_layout(); plt.show()


## 3. Multi-head attention

Một "đầu attention" chỉ học được một loại quan hệ. Multi-head: chạy attention $h$ lần song song với $h$ bộ $(W_Q, W_K, W_V)$ khác nhau, rồi concat lại.

$$
\text{MultiHead}(Q, K, V) = [\text{head}_1; \dots; \text{head}_h] W_O
$$

Mỗi head có thể chú ý vào khía cạnh khác: đầu này nhìn quan hệ chủ-vị, đầu kia nhìn quan hệ tính từ-danh từ, v.v. Đây là cách Transformer học được nhiều loại phụ thuộc cùng lúc.

## 4. Positional Encoding

Self-attention không có khái niệm thứ tự token — nó coi câu như một *bag of vectors*. "Tôi yêu em" và "Em yêu tôi" có cùng tập token, nhưng nghĩa khác hẳn.

Giải pháp: cộng vào mỗi token một **positional encoding** chứa thông tin về vị trí của nó trong câu. Bài báo gốc dùng sin/cos:
$$PE_{(pos, 2i)} = \sin(pos / 10000^{2i/d}), \quad PE_{(pos, 2i+1)} = \cos(pos / 10000^{2i/d})$$

Các phiên bản sau (BERT, GPT) dùng positional embedding học được thay vì sin/cos. Cả hai đều hoạt động tốt.

## 5. Một block Transformer Encoder

Ghép lại, một block encoder có dạng:
```
x → Multi-head Self-Attention → Add & LayerNorm → FFN → Add & LayerNorm → output
```

Trong đó:
- **FFN** (Feed-Forward Network): MLP 2 tầng, áp dụng *cùng cách* lên từng token độc lập.
- **Add & LayerNorm**: residual connection + layer normalization, giúp train sâu mà không bị mất gradient.

Stack nhiều block (12 cho BERT-base, 24 cho BERT-large, 96 cho GPT-3) → một mô hình ngôn ngữ mạnh.

### Trực quan positional encoding sin/cos

Mỗi vị trí được mã hoá thành một vector mà các chiều có tần số sóng khác nhau.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Sinusoidal positional encoding như trong paper gốc.
seq_len, d_model = 50, 32
pe = np.zeros((seq_len, d_model))
for pos in range(seq_len):
    for i in range(0, d_model, 2):
        pe[pos, i]   = np.sin(pos / 10000 ** (i / d_model))
        pe[pos, i+1] = np.cos(pos / 10000 ** (i / d_model))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Heatmap toàn bộ
im = axes[0].imshow(pe, aspect='auto', cmap='RdBu')
axes[0].set_xlabel('Chiều embedding'); axes[0].set_ylabel('Vị trí trong câu')
axes[0].set_title('Positional encoding (sin/cos) — heatmap')
plt.colorbar(im, ax=axes[0])

# Vài chiều cụ thể
for d in [0, 2, 6, 14]:
    axes[1].plot(pe[:, d], label=f'dim {d}')
axes[1].set_xlabel('Vị trí'); axes[1].set_ylabel('Giá trị')
axes[1].set_title('Vài chiều cụ thể của PE — sóng có tần số khác nhau')
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()


### Sơ đồ một block encoder



In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(figsize=(7, 9))
ax.set_xlim(0, 7); ax.set_ylim(0, 11); ax.axis('off')

def block(y, text, color, h=0.8):
    p = FancyBboxPatch((1.5, y), 4, h, boxstyle='round,pad=0.05',
                       facecolor=color, edgecolor='black', linewidth=1.2)
    ax.add_patch(p)
    ax.text(3.5, y + h/2, text, ha='center', va='center', fontsize=11)

def arrow(y1, y2):
    ax.add_patch(FancyArrowPatch((3.5, y1), (3.5, y2), arrowstyle='->',
                                 mutation_scale=15, color='black'))

def residual(y_start, y_end):
    ax.plot([0.7, 0.7], [y_start, y_end], 'k--', linewidth=1)
    ax.plot([0.7, 1.5], [y_end, y_end], 'k--', linewidth=1)
    ax.text(0.5, (y_start + y_end)/2, 'residual', rotation=90, fontsize=9, ha='center')

# Bottom up
ax.text(3.5, 0.3, 'Input embedding + Positional encoding', ha='center', fontsize=10, style='italic')
arrow(0.6, 1.0)

block(1.0, 'Multi-Head Self-Attention', '#d1ecf1')
arrow(1.8, 2.2)
block(2.2, 'Add & LayerNorm', '#fff3cd')
arrow(3.0, 3.4)
residual(1.0, 2.6)

block(3.4, 'Feed-Forward (FFN)', '#d4edda')
arrow(4.2, 4.6)
block(4.6, 'Add & LayerNorm', '#fff3cd')
arrow(5.4, 5.8)
residual(3.4, 5.0)

ax.text(3.5, 6.0, '(stack lặp lại N lần)', ha='center', fontsize=10, style='italic')
ax.text(3.5, 6.5, '↑', ha='center', fontsize=20)
block(6.8, 'Block 2', '#e2e3e5', h=0.6)
block(7.7, 'Block 3 ... Block N', '#e2e3e5', h=0.6)
arrow(8.3, 9.0)
block(9.0, 'Output (B, T, d)', '#cce5ff')

ax.set_title('Một block Transformer Encoder (stack N block)', fontsize=12)
plt.tight_layout(); plt.show()


# THỰC HÀNH 1: Code self-attention từ đầu

Để hiểu rõ, ta tự code một lớp Self-Attention từ scratch (single-head, không dùng `nn.MultiheadAttention`).

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.embed_dim = embed_dim
        self.W_q = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_k = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_v = nn.Linear(embed_dim, embed_dim, bias=False)

    def forward(self, x, return_attn=False):
        # x: (batch, seq_len, embed_dim)
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        # Tính ma trận tương đồng. transpose(-2, -1) đổi 2 chiều cuối.
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.embed_dim)
        attn = F.softmax(scores, dim=-1)             # (batch, seq_len, seq_len)
        out = attn @ V                               # (batch, seq_len, embed_dim)

        if return_attn:
            return out, attn
        return out

# Test trên một câu giả lập 5 token, mỗi token là vector 8 chiều.
sa = SelfAttention(embed_dim=8)
x = torch.randn(1, 5, 8)
out, attn = sa(x, return_attn=True)
print('Input shape :', x.shape)
print('Output shape:', out.shape)
print('Attn shape  :', attn.shape, '  (mỗi token ↔ mỗi token)')
print('Attn[0]     =\n', attn[0])
print('Tổng mỗi hàng (phải = 1):', attn[0].sum(dim=-1))

### Trực quan ma trận attention

Ma trận attention shape `(seq_len, seq_len)` — phần tử $(i, j)$ là "token $i$ chú ý bao nhiêu vào token $j$". Mỗi hàng cộng lại bằng 1 (do softmax).

Trên mạng chưa train, ma trận này gần như đều — chưa học được gì có ý nghĩa. Sau khi train, ta sẽ thấy các pattern thú vị (ví dụ: đại từ chú ý vào danh từ chủ, động từ chú ý vào subject).

In [ ]:
tokens = ['The', 'cat', 'sat', 'on', 'mat']
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(attn[0].detach().numpy(), cmap='Blues')
ax.set_xticks(range(5)); ax.set_yticks(range(5))
ax.set_xticklabels(tokens); ax.set_yticklabels(tokens)
ax.set_xlabel('Key'); ax.set_ylabel('Query')
ax.set_title('Ma trận attention (chưa train)')
plt.colorbar(im); plt.tight_layout(); plt.show()

# THỰC HÀNH 2: Phân loại văn bản với TransformerEncoder

Build một classifier nhỏ dùng `nn.TransformerEncoderLayer` của PyTorch trên một dataset đồ chơi — phân loại câu là tích cực hay tiêu cực.

Trong thực tế, ta hiếm khi train Transformer từ đầu (cần GPU lớn + dữ liệu khổng lồ). Nhưng bài này giúp em làm quen với API.

In [ ]:
# Mở rộng vocab và data so với bài LSTM, để có đủ tín hiệu.
from torch.nn.utils.rnn import pad_sequence

vocab = {'<pad>': 0, 'i': 1, 'you': 2, 'he': 3, 'we': 4, 'they': 5,
         'love': 6, 'like': 7, 'enjoy': 8, 'hate': 9, 'dislike': 10,
         'this': 11, 'that': 12, 'movie': 13, 'song': 14, 'food': 15,
         'place': 16, 'good': 17, 'bad': 18, 'great': 19, 'awful': 20,
         'not': 21, 'is': 22, 'so': 23, 'really': 24}

raw_data = [
    ('i love this movie',         1),
    ('we enjoy that song',         1),
    ('this food is great',         1),
    ('he likes the place',         1),  # không có 'the' trong vocab — bỏ qua khi encode
    ('they really love this',      1),
    ('i hate that movie',          0),
    ('this song is bad',           0),
    ('we dislike that food',       0),
    ('he is not good',             0),
    ('that place is awful',        0),
    ('this movie is so good',      1),
    ('i really enjoy this',        1),
    ('they hate this place',       0),
    ('that food is not great',     0),
    ('we love this song',          1),
    ('i dislike that movie',       0),
]

def encode(s):
    return [vocab[w] for w in s.lower().split() if w in vocab]

seqs   = [torch.tensor(encode(s), dtype=torch.long) for s, _ in raw_data]
labels = torch.tensor([y for _, y in raw_data], dtype=torch.long)
lengths = torch.tensor([len(s) for s in seqs])
X = pad_sequence(seqs, batch_first=True, padding_value=0)   # (N, max_len)
print('X shape:', X.shape, ' labels shape:', labels.shape)

In [ ]:
class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=32, num_heads=4, ffn_dim=64,
                 num_layers=2, num_classes=2, max_len=20):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        # Positional embedding học được — đơn giản hơn sin/cos, dùng cho seq ngắn.
        self.pos_emb   = nn.Embedding(max_len, embed_dim)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads,
            dim_feedforward=ffn_dim, dropout=0.1,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Linear(embed_dim, num_classes)

    def forward(self, x, src_key_padding_mask=None):
        # x: (B, T)
        positions = torch.arange(x.size(1), device=x.device).unsqueeze(0)
        emb = self.token_emb(x) + self.pos_emb(positions)
        # padding mask: True ở vị trí padding để Transformer bỏ qua.
        out = self.encoder(emb, src_key_padding_mask=src_key_padding_mask)
        # Lấy trung bình theo seq dimension (loại bỏ vị trí padding khi tính trung bình).
        if src_key_padding_mask is not None:
            mask = (~src_key_padding_mask).float().unsqueeze(-1)
            pooled = (out * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        else:
            pooled = out.mean(dim=1)
        return self.fc(pooled)

model = TransformerClassifier(vocab_size=len(vocab)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Tổng tham số: {n_params:,}')

In [ ]:
X, labels = X.to(device), labels.to(device)
padding_mask = (X == 0)   # True ở chỗ padding

loss_history = []
for epoch in range(200):
    model.train()
    optimizer.zero_grad()
    logits = model(X, src_key_padding_mask=padding_mask)
    loss = criterion(logits, labels)
    loss.backward()
    optimizer.step()
    loss_history.append(loss.item())
    if (epoch + 1) % 40 == 0:
        acc = (logits.argmax(1) == labels).float().mean().item()
        print(f'Epoch {epoch+1:3d}  loss = {loss.item():.4f}  acc = {acc*100:.2f}%')

plt.figure(figsize=(8, 3.5))
plt.plot(loss_history); plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.grid(alpha=0.3)
plt.title('Loss khi train Transformer classifier'); plt.show()

In [ ]:
# Predict trên câu KHÔNG có trong train.
model.eval()
test_sentences = [
    'we enjoy this song',
    'this movie is awful',
    'i hate that food',
    'they really like that place',
    'this is not good',
]
for s in test_sentences:
    ids = encode(s)
    seq = torch.tensor(ids, dtype=torch.long).unsqueeze(0).to(device)
    mask = (seq == 0)
    with torch.no_grad():
        logits = model(seq, src_key_padding_mask=mask)
        prob = F.softmax(logits, dim=1).squeeze().cpu().numpy()
    label = 'Tích cực' if prob[1] > 0.5 else 'Tiêu cực'
    print(f'"{s:30s}"  →  {label}  (P_pos = {prob[1]:.2f})')

## Tổng kết

1. Self-attention cho phép mỗi token nhìn toàn bộ câu cùng lúc — không như RNN phải đi tuần tự.
2. Multi-head attention học nhiều loại quan hệ song song.
3. Positional encoding bù lại việc self-attention không có thứ tự sẵn.
4. Toàn bộ block Transformer = self-attention + FFN + residual + LayerNorm.
5. PyTorch có sẵn `nn.TransformerEncoderLayer`, `nn.MultiheadAttention` — không cần code lại từ đầu cho bài lab này.

Trong thực tế, ta dùng pre-trained model (BERT, RoBERTa, GPT, T5) thay vì train Transformer từ đầu, vì việc đó đòi hỏi GPU và dữ liệu khổng lồ. Bài tập về nhà sẽ đưa em qua HuggingFace `transformers`.

# BÀI TẬP VỀ NHÀ

## Bài 1: Hiểu sâu self-attention

1. Tự sinh dataset đồ chơi: 200 câu độ dài 8 token, từ vocab 10 từ. Nhãn = 1 nếu từ "good" xuất hiện trong câu, ngược lại 0.
2. Train classifier với 1 layer self-attention (dùng class `SelfAttention` đã viết) + một FC layer.
3. Sau khi train, lấy ma trận attention của một câu chứa "good". Câu hỏi: token CLS có chú ý vào "good" hay không? Vẽ heatmap để minh chứng.

*Gợi ý:* để dễ phân tích, dùng "global pooling" thay vì CLS — lấy mean qua seq dim — và kiểm tra xem "good" có attention score cao trên các Query khác không.

## Bài 2: So sánh Transformer với LSTM

Lấy bài "long-dependency" ở lab LSTM (chuỗi 30 token, nhãn = ký tự đầu tiên). Build phiên bản Transformer:
- Embedding + 1-2 layer TransformerEncoder + FC.
- Train cùng số epoch như LSTM.

Báo cáo: accuracy của Transformer so với LSTM trên test. Dùng `time.time()` đo wall-clock của 1 epoch — Transformer nhanh hơn không (vì có thể tính song song)?

## Bài 3: Fine-tune BERT (nâng cao)

Sử dụng `transformers` của HuggingFace để fine-tune một mô hình **PhoBERT** (cho tiếng Việt) hoặc **DistilBERT** (cho tiếng Anh) trên một bài phân loại sentiment.

**Yêu cầu:**
1. Cài: `pip install transformers datasets accelerate`.
2. Chọn dataset:
   - **Tiếng Việt:** UIT-VSFC, hoặc tự crawl 500-1000 review sản phẩm.
   - **Tiếng Anh:** dataset `imdb` từ `datasets` library.
3. Tokenize bằng tokenizer của model.
4. Fine-tune 2-3 epoch với lr nhỏ (2e-5).
5. Báo cáo accuracy + 5 ví dụ predict.

Mã ví dụ khởi đầu:
```python
from transformers import AutoTokenizer, AutoModelForSequenceClassification
tokenizer = AutoTokenizer.from_pretrained('vinai/phobert-base')
model = AutoModelForSequenceClassification.from_pretrained('vinai/phobert-base', num_labels=2)
```

Có thể dùng `Trainer` của HuggingFace để code gọn, hoặc viết training loop PyTorch tay nếu muốn hiểu sâu.

## Bài 4: Đọc paper "Attention Is All You Need"

Đọc bản gốc tại https://arxiv.org/abs/1706.03762. Trả lời ngắn:
1. Transformer ban đầu được thiết kế cho task gì? (gợi ý: không phải sentiment)
2. Encoder và Decoder khác nhau ở chỗ nào?
3. "Masked self-attention" trong Decoder để làm gì?

## Hạn nộp
20/04/2026.